# 04c. Training Models — Tuned (LR por modelo + más paciencia)

Mismo pipeline preentrenado que `04_training_models.ipynb`, con 2 ajustes basados en la revisión de esa corrida:

1. **LR distinto por arquitectura**: en la corrida anterior, ViT tiny convergió en la epoch 1 y después empeoró — señal de LR demasiado alto para hacer fine-tuning de un transformer preentrenado. Aquí CNNs (ResNet-50, EfficientNet-B0) mantienen `1e-4`, y los transformers (Swin, ViT) bajan a `3e-5`.
2. **`PATIENCE=7`** (antes 5): val tiene solo 2 rasters y es ruidoso epoch a epoch; un poco más de margen reduce el riesgo de parar justo en un mal epoch por casualidad.

Checkpoints → `data/06_models/tuned/` (carpeta separada, no pisa ni la corrida original ni la de scratch).

Resultados en texto/CSV → `data/08_reporting/*_tuned.*`

## 1. Configuración

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data.dataset import create_dataloaders
from src.models.train import train_model
from src.utils.helpers import get_device, set_seed

DATA_DIR = ROOT / "data"
MODELS_DIR = DATA_DIR / "06_models" / "tuned"
FIGURES_DIR = ROOT / "reports" / "figures"
REPORT_DIR = DATA_DIR / "08_reporting"

BATCH_SIZE = 32
NUM_WORKERS = 4
MAX_EPOCHS = 30
PATIENCE = 7
LR_CNN = 1e-4
LR_TRANSFORMER = 3e-5
SEED = 42

MODELS = [
    {"name": "efficientnet_b0",              "label": "EfficientNet-B0 (tuned)", "lr": LR_CNN},
    {"name": "resnet50",                     "label": "ResNet-50 (tuned)",      "lr": LR_CNN},
    {"name": "swin_tiny_patch4_window7_224", "label": "Swin Transformer (tuned)", "lr": LR_TRANSFORMER},
    {"name": "vit_tiny_patch16_224",         "label": "ViT tiny (tuned)",       "lr": LR_TRANSFORMER},
]

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
device = get_device()
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(device)

print(f"Device: {device}")
print(f"GPU: {gpu_name}")
print(f"PyTorch: {torch.__version__}")
print(f"Batch={BATCH_SIZE} | Max epochs={MAX_EPOCHS} | Patience={PATIENCE}")
print(f"LR CNN={LR_CNN} | LR Transformer={LR_TRANSFORMER}")

## 2. DataLoaders

In [ ]:
loaders = create_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

train_loader = loaders["train"]
val_loader = loaders["val"]
test_loader = loaders["test"]

print(f"Train: {len(train_loader.dataset):,} chips | {len(train_loader):,} batches")
print(f"Val:   {len(val_loader.dataset):,} chips")
print(f"Test:  {len(test_loader.dataset):,} chips (notebook 05)")

## 3. Helpers — gráficos y exportación

In [ ]:
def history_to_dataframe(summary: dict) -> pd.DataFrame:
    """Combina métricas train/val por epoch en un DataFrame."""
    train_df = pd.DataFrame(summary["history"]["train"]).add_prefix("train_")
    val_df = pd.DataFrame(summary["history"]["val"]).add_prefix("val_")
    if "train_epoch" in train_df.columns:
        train_df = train_df.rename(columns={"train_epoch": "epoch"})
    if "val_epoch" in val_df.columns:
        val_df = val_df.drop(columns=["val_epoch"])
    df = pd.concat([train_df, val_df], axis=1)
    df.insert(0, "model", summary.get("model_label", summary["model_name"]))
    df.insert(1, "timm_name", summary["model_name"])
    return df


def plot_training_curves(summary: dict, save_path: Path) -> None:
    history = summary["history"]
    label = summary.get("model_label", summary["model_name"])
    epochs = [r["epoch"] for r in history["train"]]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(epochs, [r["loss"] for r in history["train"]], label="train")
    axes[0].plot(epochs, [r["loss"] for r in history["val"]], label="val")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("epoch")

    axes[1].plot(epochs, [r["accuracy"] for r in history["train"]], label="train")
    axes[1].plot(epochs, [r["accuracy"] for r in history["val"]], label="val")
    axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("epoch")

    axes[2].plot(epochs, [r["macro_f1"] for r in history["val"]], label="val macro F1", color="tomato")
    axes[2].plot(epochs, [r["recall_com_garimpo"] for r in history["val"]], label="val recall com_garimpo")
    axes[2].set_title("Val F1 / Recall com_garimpo"); axes[2].legend(); axes[2].set_xlabel("epoch")

    plt.suptitle(f"{label} — curvas de entrenamiento", y=1.02)
    plt.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def format_epoch_line(epoch: int, max_ep: int, train_r: dict, val_r: dict, elapsed: float) -> str:
    return (
        f"Epoch {epoch:02d}/{max_ep} | "
        f"train loss {train_r['loss']:.4f} acc {train_r['accuracy']:.3f} | "
        f"val loss {val_r['loss']:.4f} acc {val_r['accuracy']:.3f} "
        f"F1 {val_r['macro_f1']:.3f} "
        f"(com recall {val_r['recall_com_garimpo']:.3f}) | "
        f"{elapsed:.0f}s"
    )

## 4. Entrenar los 4 modelos (tuned)

In [ ]:
results = []
all_epochs_dfs = []
log_lines = [
    "TRAINING MODELS (TUNED) — RESULTADOS",
    f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    f"Device: {device} | GPU: {gpu_name}",
    f"PyTorch: {torch.__version__}",
    f"Batch={BATCH_SIZE} | Max epochs={MAX_EPOCHS} | Patience={PATIENCE}",
    f"LR CNN={LR_CNN} | LR Transformer={LR_TRANSFORMER}",
    f"Train chips: {len(train_loader.dataset):,} | Val: {len(val_loader.dataset):,}",
    "",
]

for i, model_cfg in enumerate(MODELS, start=1):
    name = model_cfg["name"]
    label = model_cfg["label"]
    lr = model_cfg["lr"]

    print("\n" + "=" * 60)
    print(f"[{i}/{len(MODELS)}] Entrenando: {label} ({name}) | LR={lr}")
    print("=" * 60)

    log_lines += ["=" * 60, f"MODELO: {label} ({name}) | LR={lr}", "=" * 60, ""]

    summary = train_model(
        model_name=name,
        train_loader=train_loader,
        val_loader=val_loader,
        save_dir=MODELS_DIR,
        device=device,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        lr=lr,
    )
    summary["model_label"] = label

    # CSV por epoch de este modelo
    epoch_df = history_to_dataframe(summary)
    epoch_csv = REPORT_DIR / f"epochs_{name}_tuned.csv"
    epoch_df.to_csv(epoch_csv, index=False)
    all_epochs_dfs.append(epoch_df)

    # Líneas de texto por epoch
    for tr, va in zip(summary["history"]["train"], summary["history"]["val"]):
        log_lines.append(format_epoch_line(
            tr["epoch"], MAX_EPOCHS, tr, va, tr.get("time_s", 0)
        ))

    val_history = summary["history"]["val"]
    best_val = next(r for r in val_history if r["epoch"] == summary["best_epoch"])

    log_lines += [
        "",
        f"→ Mejor epoch: {summary['best_epoch']} | val macro F1: {summary['best_val_macro_f1']:.4f}",
        f"→ val accuracy: {best_val['accuracy']:.4f} | val recall com_garimpo: {best_val['recall_com_garimpo']:.4f}",
        f"→ Checkpoint: {summary['checkpoint']}",
        f"→ CSV epochs: {epoch_csv}",
        "",
    ]

    plot_training_curves(summary, FIGURES_DIR / f"{name}_tuned_training_curves.png")

    results.append({
        "model": label,
        "timm_name": name,
        "lr": lr,
        "best_epoch": summary["best_epoch"],
        "epochs_run": len(summary["history"]["train"]),
        "val_macro_f1": round(summary["best_val_macro_f1"], 4),
        "val_accuracy": round(best_val["accuracy"], 4),
        "val_recall_com_garimpo": round(best_val["recall_com_garimpo"], 4),
        "val_f1_com_garimpo": round(best_val["f1_com_garimpo"], 4),
        "val_f1_sem_garimpo": round(best_val["f1_sem_garimpo"], 4),
        "checkpoint": summary["checkpoint"],
    })

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nEntrenamiento (tuned) completado.")

## 5. Guardar resumen (CSV + TXT)

In [ ]:
results_df = pd.DataFrame(results).sort_values("val_macro_f1", ascending=False)
display(results_df)

# CSV resumen comparativo
summary_csv = REPORT_DIR / "training_results_summary_tuned.csv"
results_df.to_csv(summary_csv, index=False)

# CSV todas las epochs de los 4 modelos
all_epochs_csv = REPORT_DIR / "training_epochs_all_models_tuned.csv"
pd.concat(all_epochs_dfs, ignore_index=True).to_csv(all_epochs_csv, index=False)

# TXT legible
best = results_df.iloc[0]
log_lines += [
    "=" * 60,
    "RESUMEN COMPARATIVO (validation, tuned)",
    "=" * 60,
    "",
    results_df.to_string(index=False),
    "",
    f"MEJOR MODELO (TUNED): {best['model']} | val macro F1 = {best['val_macro_f1']}",
    f"Checkpoint: {best['checkpoint']}",
]

summary_txt = REPORT_DIR / "training_results_tuned.txt"
summary_txt.write_text("\n".join(log_lines), encoding="utf-8")

print(f"\nArchivos guardados:")
print(f"  {summary_csv}")
print(f"  {all_epochs_csv}")
print(f"  {summary_txt}")
print(f"\nMejor modelo (tuned): {best['model']} (val F1={best['val_macro_f1']})")

## 6. Comparación rápida: original vs. tuned

In [ ]:
original_csv = REPORT_DIR / "training_results_summary.csv"

if original_csv.exists():
    orig_df = pd.read_csv(original_csv)[["timm_name", "val_macro_f1", "val_accuracy"]]
    orig_df = orig_df.rename(columns={"val_macro_f1": "val_macro_f1_original",
                                       "val_accuracy": "val_accuracy_original"})

    tuned_df = results_df[["timm_name", "val_macro_f1", "val_accuracy"]].rename(
        columns={"val_macro_f1": "val_macro_f1_tuned", "val_accuracy": "val_accuracy_tuned"}
    )

    compare = orig_df.merge(tuned_df, on="timm_name", how="outer")
    compare["f1_gain_tuned_minus_original"] = (
        compare["val_macro_f1_tuned"] - compare["val_macro_f1_original"]
    ).round(4)
    display(compare.sort_values("f1_gain_tuned_minus_original", ascending=False))

    compare_csv = REPORT_DIR / "original_vs_tuned_comparison.csv"
    compare.to_csv(compare_csv, index=False)
    print(f"\nGuardado: {compare_csv}")
else:
    print("No se encontró training_results_summary.csv (corrida original de 04_training_models.ipynb).")

## 7. Próximo paso

Evaluar estos checkpoints (`data/06_models/tuned/`) en **test** con `05_evaluation.ipynb` — ya está preparado para buscar `.pt` también dentro de subcarpetas.